# 01r Rydberg quench — PEPSKit iPEPS (bulk limit)

This notebook simulates a **quench of the three-level `01r` system** (`|0>`, `|1>`, `|r>`)
with the `backend="pepskit"` iPEPS backend. PEPSKit.jl is an **infinite-PEPS** library, so
this models the **infinite-lattice (bulk) limit** with a translation-invariant `2x2` unit
cell evolved by real-time simple update (`exp(-iH dt)`) and measured with CTMRG. It is *not*
a finite open-boundary 3x3 system like `run_quench_benchmark.ipynb` — it answers the
thermodynamic-limit question instead.

The quench: ramp the Rydberg drive `Omega_R` up and sweep the Rydberg detuning `Delta_R`
across resonance, measuring the bulk occupations `<n_r>(t)` and `<n_1>(t)`. A second cell
runs a pure hyperfine Rabi (`Omega_hf` only) as a `d=3` sanity check against `sin^2`.


In [ ]:
# Pin BLAS/OpenMP to 1 thread BEFORE importing numpy/scipy/tenpy.
# Small TN tensors (DMRG/TDVP/PEPS) => multi-threaded BLAS oversubscribes
# cores and runs 10-40x SLOWER on a loaded box. This cell must stay first
# and run before any import: the libraries read these variables only once,
# at import time.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")
print("BLAS/OpenMP threads pinned to", os.environ.get("OMP_NUM_THREADS"))

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import ryd_gate as rg
from ryd_gate import InteractionSpec
from ryd_gate.lattice import Register
from ryd_gate.protocols.digital_analog import DigitalAnalogProtocol

# Experimental-style parameters (rad/s, s, um)
a_um = 6.8                          # lattice spacing
Omega_R = 2 * np.pi * 3.8e6        # |1>-|r> Rabi (peak), rad/s
C6_70s = 2 * np.pi * 874e9         # rad/s * um^6 (Rb 70S typical)
delta_start = -2 * np.pi * 8.0e6   # Rydberg detuning sweep start, rad/s
delta_end = 2 * np.pi * 8.0e6      # ... sweep end
t_sweep = 1.5e-6                   # total quench time, s
omega_ramp_frac = 0.1             # fraction of t_sweep used to ramp Omega_R

In [ ]:
# Local compatibility shim: the kernel DigitalAnalogProtocol is now function-
# based (omega_R_fn(t), ...). These notebooks keep their piecewise-constant
# Segment schedules and lower them to step functions here. Note the backend
# integrates on n_steps uniform slices, so segment edges should be resolved
# by a generous n_steps (the notebooks already scale it with len(segments)).
from dataclasses import dataclass

import numpy as np


@dataclass
class Segment:
    duration: float
    omega_R: "float | np.ndarray" = 0.0
    omega_hf: "float | np.ndarray" = 0.0
    delta_R: "float | np.ndarray" = 0.0
    delta_hf: "float | np.ndarray" = 0.0


def segments_protocol(segments, n_steps=200):
    """Build the function-based DigitalAnalogProtocol from constant segments."""
    segments = list(segments)
    edges = np.cumsum([0.0] + [float(seg.duration) for seg in segments])

    def step_fn(name):
        values = [getattr(seg, name) for seg in segments]

        def fn(t):
            j = int(np.clip(np.searchsorted(edges, t, side="right") - 1, 0, len(values) - 1))
            return values[j]

        return fn

    protocol = DigitalAnalogProtocol(
        t_gate=float(edges[-1]),
        omega_R_fn=step_fn("omega_R"),
        omega_hf_fn=step_fn("omega_hf"),
        delta_R_fn=step_fn("delta_R"),
        delta_hf_fn=step_fn("delta_hf"),
        n_steps=n_steps,
    )
    protocol.segments = segments  # keep the schedule inspectable for plotting
    return protocol


In [ ]:
# Piecewise-constant segments approximating: ramp Omega_R up, then sweep Delta_R across
# resonance. Omega_hf = 0 here, so |0> is a spectator level for the main quench.
n_seg = 60
eps = np.finfo(float).eps
segments = []
for k in range(n_seg):
    tc = (k + 0.5) * t_sweep / n_seg
    ramp = 1.0 if omega_ramp_frac == 0 else min(1.0, tc / (omega_ramp_frac * t_sweep))
    omega_R = Omega_R * ramp
    frac = np.clip((tc - omega_ramp_frac * t_sweep) / max(t_sweep * (1 - omega_ramp_frac), eps), 0.0, 1.0)
    delta_R = delta_start + (delta_end - delta_start) * frac
    segments.append(Segment(duration=t_sweep / n_seg, omega_R=omega_R, delta_R=delta_R))
protocol = segments_protocol(segments)

# 2x2 square lattice defines the bulk (NN VdW) parameters; level structure 01r.
geom = Register.rectangle(2, 2, spacing_um=a_um)
system = rg.RydbergSystem.from_lattice(
    geom, "01r",
    interaction=InteractionSpec(C6=C6_70s, mode="nn"),   # nearest-neighbour blockade
    protocol=protocol,
)
t_eval = np.linspace(0.0, t_sweep, 7)
dt_tn = 0.2 / Omega_R   # Trotter step (same time units as t_sweep / t_eval)
V_nn = C6_70s / a_um**6
print(f"t_gate = {t_sweep*1e6:.3f} us, n_steps ~ {int(np.ceil(t_sweep/dt_tn))}, V_nn/Omega_R = {V_nn/Omega_R:.2f}")

## Run the bulk 01r quench

`unit_cell="uniform"` keeps the iPEPS translation-invariant (symmetric bulk dynamics).
`bond_dim` (D) is the iPEPS virtual bond dimension; `env_dim` (chi) the CTMRG environment.
The first call pays Julia/PEPSKit cold-start JIT (tens of seconds).

In [ ]:
_t0 = time.perf_counter()
res = rg.simulate(
    system, [], "all_ground",
    backend="pepskit",
    t_eval=t_eval,
    observables=["n_r", "n_1", "n_0"],
    backend_options={"unit_cell": "uniform", "bond_dim": 3, "env_dim": 16, "dt": dt_tn, "timeout": 1800},
)
elapsed = time.perf_counter() - _t0

n_r = np.asarray(res.metadata["obs"]["n_r"])
n_1 = np.asarray(res.metadata["obs"]["n_1"])
n_0 = np.asarray(res.metadata["obs"]["n_0"])
times = np.asarray(res.times)

print(f"PEPSKit iPEPS elapsed: {elapsed:.1f} s  (bond_dim={res.metadata['bond_dim']}, env_dim={res.metadata['env_dim']})")
print("t(us)    <n_r>    <n_1>    <n_0>")
for t, a, b, c in zip(times * 1e6, n_r, n_1, n_0):
    print(f"{t:6.3f}  {a:7.4f}  {b:7.4f}  {c:7.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.0), constrained_layout=True)
ax.plot(times * 1e6, n_r, "o-", label=r"$\langle n_r \rangle$")
ax.plot(times * 1e6, n_1, "s-", label=r"$\langle n_1 \rangle$")
ax.plot(times * 1e6, n_0, "^-", label=r"$\langle n_0 \rangle$")
ax.set_xlabel("time (us)")
ax.set_ylabel("bulk occupation")
ax.set_title("01r Rydberg sweep quench — PEPSKit iPEPS bulk")
ax.set_ylim(-0.02, 1.02)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## d=3 sanity check: pure hyperfine Rabi

With only the hyperfine drive `Omega_hf` on (and `V=0`), the bulk population oscillates
between `|1>` and `|0>` as `<n_0>(t) = sin^2(Omega_hf t / 2)`. This exercises the third
level explicitly and validates the `d=3` iPEPS path against the analytic single-site result.

In [ ]:
Omega_hf = 2 * np.pi * 2.0e6
hf_proto = DigitalAnalogProtocol.constant(omega_hf=Omega_hf, t_gate=1.0e-6)
hf_system = rg.RydbergSystem.from_lattice(
    Register.rectangle(2, 2, spacing_um=a_um), "01r",
    interaction=InteractionSpec(C6=0.0, mode="nn"),   # V=0 isolates the single-site Rabi
    protocol=hf_proto,
)
hf_t = np.linspace(0.0, 1.0e-6, 9)
hf_res = rg.simulate(
    hf_system, [], "all_ground", backend="pepskit", t_eval=hf_t, observables=["n_0", "n_1"],
    backend_options={"unit_cell": "uniform", "bond_dim": 2, "env_dim": 12, "dt": 0.2 / Omega_hf, "timeout": 1800},
)
hf_times = np.asarray(hf_res.times)
hf_n0 = np.asarray(hf_res.metadata["obs"]["n_0"])
analytic = np.sin(Omega_hf * hf_times / 2) ** 2

fig, ax = plt.subplots(figsize=(6.2, 4.0), constrained_layout=True)
ax.plot(hf_times * 1e6, hf_n0, "o", ms=7, label=r"PEPSKit $\langle n_0 \rangle$")
ax.plot(hf_times * 1e6, analytic, "-", label=r"$\sin^2(\Omega_{hf} t / 2)$")
ax.set_xlabel("time (us)")
ax.set_ylabel(r"$\langle n_0 \rangle$")
ax.set_title("Hyperfine Rabi |1> -> |0>  (3-level check)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print("max |<n_0> - sin^2| =", float(np.max(np.abs(hf_n0 - analytic))))